# 2. Conditional Edges / Branching

`add_conditional_edges` routes execution to a different node based on the
current state, instead of a fixed next step. Demonstrated with a small router:
classify a question as "math" or "general", then send it to a node specialized
for that category.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph

from models.chat_models.ollama_models import SupportedModel, get_chat_model
from tools.math_tools import adder, divider, multiplier, subtractor

MATH_TOOLS = [adder, subtractor, multiplier, divider]
MATH_TOOLS_BY_NAME = {tool.__name__: tool for tool in MATH_TOOLS}


class RouterState(TypedDict):
    question: str
    category: str
    answer: str


llm = get_chat_model(SupportedModel.llama3_2)


def classify(state: RouterState) -> dict:
    response = llm.invoke(
        "Classify the following question as exactly one word, either "
        "'math' or 'general' — respond with nothing else.\n\n"
        f"Question: {state['question']}"
    )
    category = "math" if "math" in response.content.lower() else "general"
    return {"category": category}


def route(state: RouterState) -> Literal["solve_math", "answer_general"]:
    return "solve_math" if state["category"] == "math" else "answer_general"


def solve_math(state: RouterState) -> dict:
    ai_message = llm.bind_tools(MATH_TOOLS).invoke(state["question"])
    if not ai_message.tool_calls:
        return {"answer": ai_message.content}
    tool_call = ai_message.tool_calls[0]
    result = MATH_TOOLS_BY_NAME[tool_call["name"]](**tool_call["args"])
    return {"answer": f"{tool_call['name']}({tool_call['args']}) = {result}"}


def answer_general(state: RouterState) -> dict:
    response = llm.invoke(state["question"])
    return {"answer": response.content}


graph = StateGraph(RouterState)
graph.add_node("classify", classify)
graph.add_node("solve_math", solve_math)
graph.add_node("answer_general", answer_general)
graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route)
graph.add_edge("solve_math", END)
graph.add_edge("answer_general", END)
compiled = graph.compile()

## A math question

In [ ]:
math_result = compiled.invoke({"question": "what is 12 times 7?", "category": "", "answer": ""})
print("category:", math_result["category"])
print("answer:", math_result["answer"])

## A general-knowledge question

In [ ]:
general_result = compiled.invoke({"question": "why is the sky blue?", "category": "", "answer": ""})
print("category:", general_result["category"])
print("answer:", general_result["answer"])

## 🧪 Playground

**1. A third category** — add `"creative"` (writing prompts), a `write_creative` node, and update `classify`'s prompt + `route`'s `Literal` return type.

In [ ]:
# TODO: add a third branch and rewire the graph


**2. Force a misclassification** — try an ambiguous question (e.g. `"How many sides does a stop sign have?"` — arguably math, arguably general) and see which way `classify` routes it. (Documented Gotcha: `classify` is a separate LLM call from either specialist, so ambiguous input can be routed to the "wrong" specialist.)

In [ ]:
# TODO: try an ambiguous question and inspect category + answer


**3. Visualize the graph** — `print(compiled.get_graph().draw_mermaid())`.

In [ ]:
# TODO: draw_mermaid()
